# Latent-Play: A Playable World Model Learned from Action-Free Video

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rivas-vega/latent-play/blob/main/notebooks/colab_training_and_demo.ipynb)

Este notebook ejecuta el flujo completo de **`latent-play`** en Google Colab Pro (**A100 GPU + High-RAM**):
1. **Setup + Verificación de Hardware + Tests Unitarios** (41 tests)
2. **Fase A — Validación Rápida en `ToyGridEnv` (~3 min en A100)**: verifica de punta a punta LAM $\to$ JEPA Dynamics + SIGReg $\to$ CEM Planning antes del entrenamiento largo.
3. **Fase B — Benchmark Principal en `Crafter` (A100)**:
   - Recolección de 300k transiciones
   - Entrenamiento de **LAM** sin etiquetas (`label_fraction=0.0`)
   - Entrenamiento de **JEPA Dynamics + SIGReg** (con acciones latentes y Oráculo 100% supervisado)
   - **Curva de Eficiencia de Etiquetas** (`0.1%`, `1%`, `5%`, `10%`, `100%`) y **Planificación CEM**
   - **Violation-of-Expectation (VoE)** Physics Scorer y generación de figuras finales

## 1. Setup, Clonado del Repo y Verificación de A100

In [ ]:
import os, torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Si estás en Colab y aún no estás dentro de la carpeta del repo, clónalo o actualiza:
# (Si el repo es privado, puedes usar un Personal Access Token de GitHub o subir el zip)
import os
if not os.path.exists("latent_play"):
    if os.path.exists("latent-play"):
        %cd latent-play
        !git pull origin main
    else:
        print("Clonando repositorio... Si es privado, introduce tu token o sube la carpeta:")
        !git clone https://github.com/rivas-vega/latent-play.git && cd latent-play
        if os.path.exists("latent-play"):
            %cd latent-play
else:
    !git pull origin main

!pip install -q -e ".[crafter,dev]" matplotlib
!pytest -q

## 2. Fase A — Sanity Check Completo sobre `ToyGridEnv` (~3 min en A100)
Ejecutamos primero el pipeline completo sobre `ToyGridEnv` (agente rojo + distractor azul con paseo aleatorio + fondo cambiante) para comprobar en pocos minutos que:
- El **LAM** ignora el distractor y descubre las acciones reales (`val/purity_visible` alta).
- **JEPA Dynamics + SIGReg** predice rollouts multi-paso sin colapso dimensional (`eff_rank` alto).
- **CEM Planner** navega hacia objetivos visuales.

In [ ]:
!python scripts/collect_data.py --config configs/collect_toy.yaml --total-steps 100000 --workers 4
!python scripts/train_lam.py --config configs/lam_toy.yaml train.amp=auto
!python scripts/train_dyn.py --config configs/dyn_toy.yaml lam_checkpoint=checkpoints/lam_toy/lam_final.pt train.out_dir=checkpoints/dyn_toy_lam train.amp=auto
!python scripts/train_dyn.py --config configs/dyn_toy.yaml train.out_dir=checkpoints/dyn_toy_oracle train.amp=auto
!python scripts/sweep_labels.py --data-root data/toy --lam-ckpt checkpoints/lam_toy/lam_final.pt --dyn-ckpt checkpoints/dyn_toy_lam/dyn_final.pt --oracle-ckpt checkpoints/dyn_toy_oracle/dyn_final.pt --fractions 0.001 0.01 0.05 0.1 1.0 --num-episodes 25 --out-dir outputs/sweep_toy

## 3. Fase B — Benchmark Principal en `Crafter` (300k pasos)

In [ ]:
!python scripts/collect_data.py --config configs/collect_crafter.yaml --total-steps 300000 --workers 4
!python scripts/inspect_data.py --data data/crafter --out outputs/crafter_samples.gif

## 4. Entrenar Latent Action Model (LAM) en `Crafter` (`label_fraction=0.0`)

In [ ]:
!python scripts/train_lam.py --config configs/lam_crafter.yaml train.compile=true train.amp=auto

## 5. Entrenar JEPA Dynamics + SIGReg en `Crafter` (Latent Actions vs. Oracle)

In [ ]:
# 5a. Nuestro modelo (latent-play): condicionado únicamente en códigos latentes del LAM
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml lam_checkpoint=checkpoints/lam_crafter/lam_final.pt train.out_dir=checkpoints/dyn_crafter_lam train.amp=auto

# 5b. Oráculo (Upper Bound): condicionado en el 100% de las acciones verdaderas de Crafter
!python scripts/train_dyn.py --config configs/dyn_crafter.yaml train.out_dir=checkpoints/dyn_crafter_oracle train.amp=auto

## 6. Barrido de Fracción de Etiquetas (0.1% a 100%), Planificación CEM y VoE Scorer

In [ ]:
!python scripts/sweep_labels.py \
    --data-root data/crafter \
    --lam-ckpt checkpoints/lam_crafter/lam_final.pt \
    --dyn-ckpt checkpoints/dyn_crafter_lam/dyn_final.pt \
    --oracle-ckpt checkpoints/dyn_crafter_oracle/dyn_final.pt \
    --fractions 0.001 0.01 0.05 0.1 1.0 \
    --num-episodes 30 \
    --out-dir outputs/sweep_results

!python scripts/eval_voe.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --lam-checkpoint checkpoints/lam_crafter/lam_final.pt \
    --out outputs/voe_metrics.json

!python scripts/plot_paper_figures.py --metrics-json outputs/sweep_results/sweep_metrics.json --out-dir outputs/figures

In [ ]:
!python scripts/eval_planning.py \
    --config configs/dyn_crafter.yaml \
    --checkpoint checkpoints/dyn_crafter_lam/dyn_final.pt \
    --alignment-checkpoint outputs/sweep_results/alignment_f0.01_s0.pt \
    --num-episodes 20

from IPython.display import Image
Image("outputs/eval_planning/planned_trajectory.gif")

## 7. Empaquetar Resultados y Logs para Descargar

In [ ]:
!zip -r latent_play_results.zip outputs/ checkpoints/*/log.jsonl checkpoints/*/confusion.json checkpoints/*/config.json
from google.colab import files
files.download("latent_play_results.zip")